# Logits Alignments

In [1]:
def default_params(): 
    return {
        'current_model': 'M1',
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/datax/code_smells/generation',
            #['curated','greedy_search', 'beam_search', 'sampling', 'contrastive_search', 'top_k_sampling', 'top_p_sampling']
            'decoding_strategy': 'top_p_sampling',
            'content_column': 'code',
            'sampling_size': 500,
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'raw_logits_path' : '/workspaces/CodeSmells/datax/code_smells/logits/generation',
        'alignments_path': '/workspaces/CodeSmells/data/extension/generation/alignments',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'microsoft/Phi-3.5-mini-instruct', #https://huggingface.co/microsoft/Phi-3.5-mini-instruct 
            'M4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B
            'M5' : 'facebook/incoder-6B', #https://huggingface.co/facebook/incoder-6B
            'M6' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b 
            'M7' : 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
            'M8' : 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import random
import numpy as np
from statistics import mean, median
import os
import torch
import gc
from difflib import SequenceMatcher
from scipy.stats import entropy

In [3]:
from datasets import load_dataset, Dataset

In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

2025-03-27 20:43:24.014917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743108204.032887 1968158 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743108204.038508 1968158 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-27 20:43:24.056658: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['decoding_strategy']}"
create_folder(log_file)
log_file += '/align_aggr.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Model Loading

In [8]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [9]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#### Load Dataset

In [10]:
df_actual_ntp = pd.read_json(f"{params['raw_logits_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}/raw_logits.json")

In [11]:
df_actual_ntp.head(2)

,id,commit_id,repo,path,file_name,commit_message,url,language,category,prompt,...,ast_levels,n_ast_nodes,n_ast_errors,n_identifiers,input_ids,input_lenght,max_prob,min_prob,actual_prob,loss
0,318705,fc695896dd8b0169001c438054a79e347053fac6,paperless-ngx,src/documents/tests/test_matchables.py,test_matchables.py,Format Python code with black,https://github.com/paperless-ngx/paperless-ngx...,Python,Warning,complete the following incomplete Python funct...,...,12,198,0,16,"[822, 731, 3373, 29898, 1311, 1125, 13, 4706, ...",224,"[[<PRE>, 0.7492889762], [module, 0.47944223880...","[[<s>, 0.0], [$}, 1e-10], [oreferrer, 0.0], [u...","[[def, 0.0007611390000000001], [set, 0.0013520...",0.865045
1,114041,f95ff4da7713a3d3c27fb56f4dcd54c63cd2a1af,mindsdb,mindsdb/api/mysql/mysql_proxy/classes/sql_stat...,sql_statement_parser.py,del aitable,https://github.com/mindsdb/mindsdb.git,Python,Warning,complete the following incomplete Python funct...,...,11,244,0,48,"[822, 679, 29918, 26766, 29898, 2850, 1125, 13...",482,"[[<PRE>, 0.7492900491000001], [module, 0.47944...","[[<s>, 0.0], [$}, 1e-10], [oreferrer, 0.0], [o...","[[def, 0.0007611316000000001], [get, 0.0267731...",0.578146


#### Token Binding

In [12]:
def find_range_of_indexes(positions, search_range):
    """
    Finds the range of indexes in the positions array where the search_range is fully included.
    
    Args:
    - positions: A list of tuples, where each tuple is (start_position, end_position) (inclusive).
    - search_range: A tuple (start_position, end_position), where start_position is inclusive and end_position is exclusive.
    
    Returns:
    - A tuple (start_index, end_index) representing the range of indexes in the positions array where the search_range is included.
    """
    start, end = search_range
    start_index = -1
    end_index = -1

    for i, (pos_start, pos_end) in enumerate(positions):
        if pos_start <= start <= pos_end:  # Find the start of the range
            start_index = i
        if pos_start <= end - 1 <= pos_end and pos_end>=end:  # Find the end of the range
            end_index = i
            break

    if start_index != -1 and end_index != -1:
        return (start_index, end_index)
    else:
        return None  # If no range is found

In [13]:
def get_substring_positions(code: str, code_smell: str, start):
    """
    Calculate the start and end positions of the substring based on line and column information.

    Parameters:
    text (str): The input string containing multiple lines.
    start (tuple): A tuple of (start_line, start_column) indicating the start position.
    end (tuple): A tuple of (end_line, end_column) indicating the end position.

    Returns:
    tuple: A tuple containing (start_position, end_position) of the substring in the input string.
    """
    lines = code.split('\n')  # Split the string into lines

    # Calculate the character position for the start of the substring
    start_line, start_column = start
    
    try:
        start_position = sum(len(lines[i]) + 1 for i in range(start_line - 1)) + start_column
    except:
        start_position = code.find(code_smell)

    if start_line > len(lines) or start_position>= len(code): 
        start_position = code.find(code_smell)

    if start_position == -1:
        match = SequenceMatcher(None, code, code_smell).find_longest_match()
        start_position= match.a
        end_position = match.a + match.size
    else:
        end_position = start_position + len(code_smell)
        end_position = len(code) if end_position >= len(code) else end_position
    

    return (start_position, end_position)

In [14]:
def find_code_smell_logits(code, code_smell_pos, logits_array, tokenizer):
    indexes_range = find_range_of_indexes(tokenizer.encode_plus(code, return_offsets_mapping=True, add_special_tokens=False)['offset_mapping'], code_smell_pos)
    if indexes_range == None: return None
    return logits_array[indexes_range[0]:indexes_range[1]+1]

In [15]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'commit_message', 'url',
       'language', 'category', 'prompt', 'original_code', 'code', 's_msg_id',
       's_line', 's_column', 's_end_line', 's_end_column', 's_code',
       'n_whitespaces', 'n_words', 'vocab_size', 'fun_name', 'complexity',
       'nloc', 'token_counts', 'ast_errors', 'ast_levels', 'n_ast_nodes',
       'n_ast_errors', 'n_identifiers', 'input_ids', 'input_lenght',
       'max_prob', 'min_prob', 'actual_prob', 'loss'],
      dtype='object')

In [16]:
df_actual_ntp['code_smell_pos'] = df_actual_ntp.apply(lambda row: get_substring_positions(row['code'], row['s_code'], (row['s_line'], row['s_column'])), axis=1)

In [17]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'commit_message', 'url',
       'language', 'category', 'prompt', 'original_code', 'code', 's_msg_id',
       's_line', 's_column', 's_end_line', 's_end_column', 's_code',
       'n_whitespaces', 'n_words', 'vocab_size', 'fun_name', 'complexity',
       'nloc', 'token_counts', 'ast_errors', 'ast_levels', 'n_ast_nodes',
       'n_ast_errors', 'n_identifiers', 'input_ids', 'input_lenght',
       'max_prob', 'min_prob', 'actual_prob', 'loss', 'code_smell_pos'],
      dtype='object')

#### Aggregation Functions

In [18]:
def compute_relative_psc(actual_probs, min_probs, max_probs, epsilon=1e-9):
    """
    Computes the adjusted PSC using normalized relative probabilities.

    Parameters:
    - actual_probs: np.array of actual softmax probabilities for tokens in a smell
    - min_probs: np.array of minimum softmax probabilities observed for each token
    - max_probs: np.array of maximum softmax probabilities observed for each token
    - epsilon: small value to avoid division by zero
    
    Returns:
    - relative PSC score
    """
    # Normalize probabilities
    relative_probs = (actual_probs - min_probs) / (max_probs - min_probs + epsilon)

    # Compute relative PSC as the mean of relative probabilities
    relative_psc = np.mean(relative_probs)
    
    return relative_psc

In [19]:
def compute_psc_entropy_scaled(actual_probs, min_probs, max_probs, temperature=1.0, epsilon=1e-9):
    """
    Computes the PSC score using entropy normalization and temperature scaling.

    Parameters:
    - actual_probs: np.array of softmax probabilities for tokens in a smell
    - min_probs: np.array of minimum softmax probabilities observed for each token
    - max_probs: np.array of maximum softmax probabilities observed for each token
    - temperature: Temperature scaling factor (default=1.0)
    - epsilon: small value to avoid division by zero
    
    Returns:
    - Adjusted PSC score
    """

    # Apply temperature scaling
    scaled_probs = np.exp(actual_probs / temperature) / np.sum(np.exp(actual_probs / temperature))

    # Compute Shannon entropy per token
    entropy_scores = entropy(scaled_probs, base=2)  # Base 2 for information entropy

    # Normalize using entropy-based weighting
    entropy_norm = 1 - (entropy_scores / np.log2(len(actual_probs) + epsilon))  # Normalize between 0 and 1

    # Compute final PSC score (weighted by entropy)
    adjusted_psc = np.mean(entropy_norm * scaled_probs)

    return adjusted_psc

#### Execute

In [20]:
df_actual_ntp.head(2)

,id,commit_id,repo,path,file_name,commit_message,url,language,category,prompt,...,n_ast_nodes,n_ast_errors,n_identifiers,input_ids,input_lenght,max_prob,min_prob,actual_prob,loss,code_smell_pos
0,318705,fc695896dd8b0169001c438054a79e347053fac6,paperless-ngx,src/documents/tests/test_matchables.py,test_matchables.py,Format Python code with black,https://github.com/paperless-ngx/paperless-ngx...,Python,Warning,complete the following incomplete Python funct...,...,198,0,16,"[822, 731, 3373, 29898, 1311, 1125, 13, 4706, ...",224,"[[<PRE>, 0.7492889762], [module, 0.47944223880...","[[<s>, 0.0], [$}, 1e-10], [oreferrer, 0.0], [u...","[[def, 0.0007611390000000001], [set, 0.0013520...",0.865045,"(521, 530)"
1,114041,f95ff4da7713a3d3c27fb56f4dcd54c63cd2a1af,mindsdb,mindsdb/api/mysql/mysql_proxy/classes/sql_stat...,sql_statement_parser.py,del aitable,https://github.com/mindsdb/mindsdb.git,Python,Warning,complete the following incomplete Python funct...,...,244,0,48,"[822, 679, 29918, 26766, 29898, 2850, 1125, 13...",482,"[[<PRE>, 0.7492900491000001], [module, 0.47944...","[[<s>, 0.0], [$}, 1e-10], [oreferrer, 0.0], [o...","[[def, 0.0007611316000000001], [get, 0.0267731...",0.578146,"(173, 180)"


In [21]:
# Alignments
df_actual_ntp['code_smell_actual_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['actual_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_max_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['max_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_min_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['min_prob'], tokenizer), axis=1)

In [22]:
df_actual_ntp = df_actual_ntp.dropna().copy()

In [23]:
## Aggregations - median
df_actual_ntp['code_smell_actual_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [24]:
## Aggregations  - mean
df_actual_ntp['code_smell_actual_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [25]:
## Aggregations - entropy scaled
df_actual_ntp['code_smell_psc_entropy'] = df_actual_ntp.apply(lambda row: compute_psc_entropy_scaled(np.array([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']])), axis=1)
## Aggregations - normalized relative probabilities
df_actual_ntp['code_smell_psc_relative'] = df_actual_ntp.apply(lambda row: compute_relative_psc(np.array([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']])), axis=1)

In [26]:
df_actual_ntp

,id,commit_id,repo,path,file_name,commit_message,url,language,category,prompt,...,code_smell_max_logits,code_smell_min_logits,code_smell_actual_prob_median,code_smell_max_prob_median,code_smell_min_prob_median,code_smell_actual_prob_mean,code_smell_max_prob_mean,code_smell_min_prob_mean,code_smell_psc_entropy,code_smell_psc_relative
0,318705,fc695896dd8b0169001c438054a79e347053fac6,paperless-ngx,src/documents/tests/test_matchables.py,test_matchables.py,Format Python code with black,https://github.com/paperless-ngx/paperless-ngx...,Python,Warning,complete the following incomplete Python funct...,...,"[[def, 0.7806562185], [xt, 0.9997345805000001]...","[[await, 0.0], [ederbörd, 0.0], [ightarrow, 1....",0.985243,0.985243,0.0,0.786325,0.937719,3.740517e-17,0.008179,0.806068
1,114041,f95ff4da7713a3d3c27fb56f4dcd54c63cd2a1af,mindsdb,mindsdb/api/mysql/mysql_proxy/classes/sql_stat...,sql_statement_parser.py,del aitable,https://github.com/mindsdb/mindsdb.git,Python,Warning,complete the following incomplete Python funct...,...,"[[CE, 0.9710325003], [,, 0.8635253906], [D, 0....","[[bolds, 0.0], [nah, 0.0], [Norweg, 1e-10], [o...",0.676999,0.676999,0.0,0.596387,0.596387,2.500000e-11,0.009867,1.000000
2,153807,c1d5dbd71efb8fb5806fad41959794182780fc25,modin,modin/logging/config.py,config.py,FEAT-#4501: Add RSS Memory Profiling to Modin ...,https://github.com/modin-project/modin.git,Python,Warning,complete the following incomplete Python funct...,...,"[[ , 0.5239624977], [mod, 0.6639267206], ...","[[oreferrer, 0.0], [float, 0.0], [usetts, 0.0]...",0.989127,0.989127,0.0,0.747203,0.803856,6.238392e-17,0.002814,0.891875
3,157787,e292b514b8f4873a36c8ca0ba68b19db2ee8ba44,d2l-zh,d2l/paddle.py,paddle.py,[Paddle] Add chapter chapter_linear-networks (...,https://github.com/d2l-ai/d2l-zh.git,Python,Convention,complete the following incomplete Python funct...,...,"[[#, 0.13408300280000002]]","[[rant, 0.0]]",0.085771,0.134083,0.0,0.085771,0.134083,0.000000e+00,1.000000,0.639683
4,18541,6c7211cbb870c97e4069c67d438e6582ffce07e1,ccxt,python/ccxt/binance.py,binance.py,1.73.14\n\n[ci skip],https://github.com/ccxt/ccxt.git,Python,Warning,complete the following incomplete Python funct...,...,"[[ , 0.8976882696], [market, 0.310562...","[[witz, 0.0], [cx, 0.0]]",0.604125,0.604125,0.0,0.604125,0.604125,0.000000e+00,0.029793,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21455,222597,8198943edd73a363c266633e1aa5b2a9e9c9f526,XX-Net,python3.10.4/Lib/distutils/ccompiler.py,ccompiler.py,add python 3.10.4 for windows,https://github.com/XX-net/XX-Net.git,Python,Warning,complete the following incomplete Python funct...,...,"[[class, 0.2209747732], [\n, 0.348399758300000...","[[oreferrer, 0.0], [oreferrer, 0.0], [<MID, 0....",0.631976,0.631976,0.0,0.563246,0.610807,0.000000e+00,0.003315,0.856204
21456,313437,f7bd88c952d398fa4e3b8d2510aa61e21db8007a,core,homeassistant/components/feedreader/__init__.py,__init__.py,Fix Feedreader Atom feeds using `updated` date...,https://github.com/home-assistant/core.git,Python,Refactor,complete the following incomplete Python funct...,...,"[[from, 0.42285889390000003], [class, 0.222695...","[[oreferrer, 0.0], [oreferrer, 0.0], [orient, ...",0.285525,0.427331,0.0,0.422316,0.507407,0.000000e+00,0.004083,0.715929
21457,222597,8198943edd73a363c266633e1aa5b2a9e9c9f526,XX-Net,python3.10.4/Lib/distutils/ccompiler.py,ccompiler.py,add python 3.10.4 for windows,https://github.com/XX-net/XX-Net.git,Python,Warning,complete the following incomplete Python funct...,...,"[[\n, 0.34839975830000003], [dist, 0.728634297...","[[oreferrer, 0.0], [<MID, 0.0], [ViewById, 0.0...",0.631976,0.631976,0.0,0.607125,0.641653,0.000000e+00,0.001820,0.900895
21458,211711,ddd8740a7b868e0496f4dae33f502b4585e58143,PaddleDetection,configs/rotate/tools/inference_benchmark.py,inference_benchmark.py,fix version of Paddle-TRT needed by ppyoloe_r ...,https://github.com/PaddlePaddle/PaddleDetectio...,Python,Warning,complete the fo

#### SAVE

In [27]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [28]:
alignments_dir = f"{params['alignments_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}"
create_folder(alignments_dir)
df_actual_ntp.to_json(f"{alignments_dir}/aligned_smells.json")

In [29]:
torch.cuda.empty_cache()
gc.collect()

0